In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

# ===================================================================
# 1. DOWNLOAD E PREPARO DOS DADOS (mesma lógica do monetinha.jl)
# ===================================================================

acoes = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "BBAS3.SA",
         "ITSA4.SA", "WEGE3.SA", "RENT3.SA", "SUZB3.SA", "ABEV3.SA"]
data_inicio = "2023-01-01"
data_fim    = "2026-06-30"

precos = pd.DataFrame()

for ticker in acoes:
    dados = yf.download(ticker, start=data_inicio, end=data_fim, progress=False)
    if dados.empty:
        print(f"Aviso: falha ao baixar {ticker}, pulando...")
        continue
    precos[ticker] = dados["Close"]
    print(f"Sucesso: {ticker}")

acoes = list(precos.columns)   # mantém só os tickers que baixaram com sucesso
n_acoes = len(acoes)
print(f"\n{n_acoes} ações carregadas com sucesso.")

matriz_precos = precos.values  # shape: (dias, n_acoes)

# Retorno diário, vetorizado — mesma conta de sempre:
# retorno = (preco_hoje - preco_ontem) / preco_ontem
retornos_diarios = (matriz_precos[1:, :] / matriz_precos[:-1, :]) - 1.0

medias       = np.mean(retornos_diarios, axis=0)         # vetor (n_acoes,)
covariancias = np.cov(retornos_diarios, rowvar=False)     # matriz (n_acoes, n_acoes)


# ===================================================================
# 2. REPRESENTAÇÃO GENÉTICA DO PORTFÓLIO
# ===================================================================
# Cada cromossomo tem n_acoes "genes" reais e LIVRES (sem restrição
# nenhuma, podem ser positivos ou negativos). Pra virar um portfólio
# válido (pesos >= 0 e soma = 1), aplicamos SOFTMAX nos genes: isso
# transforma qualquer vetor real num vetor de pesos positivos que
# soma exatamente 1 — sem nunca violar a restrição, não importa quais
# valores o gene assuma.
#
#   peso_i = exp(gene_i) / soma(exp(genes))
#
# É o mesmo truque usado em redes neurais pra transformar "scores"
# crus em probabilidades.

def criar_cromossomos(qtd_cromossomos: int, qtd_genes: int) -> np.ndarray:
    # genes começam em [-1, 1], igual ao CredituS — a faixa exata não
    # importa muito pro softmax, só precisa ter sinal variado
    return -1 + 2 * np.random.rand(qtd_cromossomos, qtd_genes)


def genes_para_pesos(genes: np.ndarray) -> np.ndarray:
    exp_genes = np.exp(genes - np.max(genes))  # subtrai o máximo só por estabilidade numérica
    return exp_genes / np.sum(exp_genes)


def calcular_fitness(cromossomos: np.ndarray, medias: np.ndarray, covariancias: np.ndarray) -> np.ndarray:
    """
    Pra cada cromossomo (linha):
      1. converte os genes em pesos válidos (soma 1, todos >= 0) via softmax
      2. calcula retorno = pesos . medias
      3. calcula risco   = pesos' * covariancias * pesos
      4. fitness = retorno / risco  (mesma função-objetivo do monetinha.jl)
    """
    lista_fitness = []

    for linha in cromossomos:
        pesos = genes_para_pesos(linha)

        retorno = np.dot(pesos, medias)
        risco   = pesos @ covariancias @ pesos   # forma quadrática w' Sigma w

        if risco > 1e-12:
            fitness = retorno / risco
        else:
            fitness = -np.inf  # carteira degenerada (risco ~0), descarta

        lista_fitness.append(fitness)

    return np.array(lista_fitness)


def fitness_percentual(vetor_fitnesses: np.ndarray) -> np.ndarray:
    # Diferença importante em relação ao CredituS: aqui o fitness PODE
    # SER NEGATIVO (retorno negativo / risco positivo = índice negativo),
    # e a roleta precisa de valores >= 0 pra funcionar. Por isso
    # deslocamos tudo pelo mínimo antes de normalizar.
    fitnesses_finitos = vetor_fitnesses[np.isfinite(vetor_fitnesses)]
    minimo = np.min(fitnesses_finitos) if len(fitnesses_finitos) > 0 else 0.0

    fitness_ajustado = np.where(np.isfinite(vetor_fitnesses), vetor_fitnesses - minimo, 0.0)
    fitness_ajustado = fitness_ajustado + 1e-9  # evita zero absoluto pra todo mundo ter alguma chance

    soma = np.sum(fitness_ajustado)
    return fitness_ajustado / soma


def selecionar_pais_roleta(cromossomos: np.ndarray, percentual_fitnesses: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    roleta_acumulada = np.cumsum(percentual_fitnesses)

    indice_pai = min(int(np.searchsorted(roleta_acumulada, np.random.rand())), len(cromossomos) - 1)
    indice_mae = min(int(np.searchsorted(roleta_acumulada, np.random.rand())), len(cromossomos) - 1)

    return cromossomos[indice_pai], cromossomos[indice_mae]


def cruzar_pais(pai: np.ndarray, mae: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    c1 = np.random.randint(1, len(pai))
    c2 = np.random.randint(1, len(pai))
    c3 = np.random.randint(1, len(pai))

    filho1 = np.concatenate([pai[:c1], mae[c1:]])
    filho2 = np.concatenate([pai[:c2], mae[c2:]])
    filho3 = np.concatenate([pai[:c3], mae[c3:]])

    return filho1, filho2, filho3


def mutar(filho1: np.ndarray, filho2: np.ndarray, filho3: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    for filho in [filho1, filho2, filho3]:
        indice = np.random.randint(0, len(filho))
        filho[indice] = -1 + 2 * np.random.rand()

    return filho1, filho2, filho3


def atualizar_populacao(cromossomos: np.ndarray, vetor_fitnesses: np.ndarray,
                         filho1: np.ndarray, filho2: np.ndarray, filho3: np.ndarray,
                         medias: np.ndarray, covariancias: np.ndarray) -> np.ndarray:
    filhos = np.array([filho1, filho2, filho3])
    fitnesses_filhos = calcular_fitness(filhos, medias, covariancias)

    indices_melhores_filhos = np.argsort(fitnesses_filhos)[-2:]  # 2 melhores filhos
    indices_piores          = np.argsort(vetor_fitnesses)[:2]    # 2 piores da população atual

    nova_populacao = cromossomos.copy()
    for i in range(2):
        idx_pior  = int(indices_piores[i])
        idx_filho = int(indices_melhores_filhos[i])
        nova_populacao[idx_pior] = filhos[idx_filho]

    return nova_populacao


def algoritmo_genetico(medias: np.ndarray, covariancias: np.ndarray, n_acoes: int,
                        qtd_cromossomos: int = 60, geracoes: int = 300,
                        geracoes_sem_melhora_max: int = 50) -> tuple[np.ndarray, float]:

    populacao = criar_cromossomos(qtd_cromossomos, n_acoes)

    melhor_cromossomo    = None
    melhor_fitness        = -np.inf
    geracoes_sem_melhora  = 0

    for geracao in range(geracoes):

        fitnesses   = calcular_fitness(populacao, medias, covariancias)
        percentuais = fitness_percentual(fitnesses)

        idx_melhor    = int(np.argmax(fitnesses))
        fitness_atual = float(fitnesses[idx_melhor])

        if fitness_atual > melhor_fitness:
            melhor_fitness       = fitness_atual
            melhor_cromossomo    = populacao[idx_melhor].copy()
            geracoes_sem_melhora = 0
        else:
            geracoes_sem_melhora += 1

        if (geracao + 1) % 20 == 0 or geracao == 0:
            print(f"Geração {geracao+1:>4} | melhor fitness: {melhor_fitness:.6f}")

        # Critério de parada: CONVERGÊNCIA (sem melhora há N gerações),
        # não um "fitness alvo" fixo como no CredituS — aqui não existe
        # um teto natural de 0 a 1 pra esse índice retorno/risco.
        if geracoes_sem_melhora >= geracoes_sem_melhora_max:
            print(f"\nConvergiu: {geracoes_sem_melhora_max} gerações sem melhora (parou na geração {geracao+1}).")
            break

        pai, mae = selecionar_pais_roleta(populacao, percentuais)
        filho1, filho2, filho3 = cruzar_pais(pai, mae)
        filho1, filho2, filho3 = mutar(filho1, filho2, filho3)
        populacao = atualizar_populacao(populacao, fitnesses, filho1, filho2, filho3, medias, covariancias)

    return melhor_cromossomo, melhor_fitness


# ===================================================================
# 3. EXECUÇÃO
# ===================================================================

melhor_cromossomo, melhor_fitness = algoritmo_genetico(medias, covariancias, n_acoes)
pesos_otimos = genes_para_pesos(melhor_cromossomo)

print("\n=== MELHOR CARTEIRA ENCONTRADA (via GA) ===")
for acao, peso in zip(acoes, pesos_otimos):
    print(f"{acao}: {peso*100:.2f}%")
print(f"Índice retorno/risco: {melhor_fitness:.4f}")

# ===================================================================
# 4. TABELA DE ALOCAÇÃO (mesma lógica do monetinha.jl)
# ===================================================================

investimento = 10_000.0
preco_atual  = matriz_precos[-1, :]   # último dia = preço mais recente

valor_por_acao = pesos_otimos * investimento
qtd_por_acao   = np.floor(valor_por_acao / preco_atual)  # arredonda pra baixo

alocacao = pd.DataFrame({
    "Acao": acoes,
    "Peso_pct": np.round(pesos_otimos * 100, 2),
    "Preco_Atual_R$": np.round(preco_atual, 2),
    "Valor_Alocado_R$": np.round(valor_por_acao, 2),
    "Quantidade": qtd_por_acao.astype(int),
    "Valor_Real_R$": np.round(qtd_por_acao * preco_atual, 2),
})

linha_total = pd.DataFrame([{
    "Acao": "TOTAL",
    "Peso_pct": 100.0,
    "Preco_Atual_R$": np.nan,
    "Valor_Alocado_R$": round(valor_por_acao.sum(), 2),
    "Quantidade": np.nan,
    "Valor_Real_R$": round((qtd_por_acao * preco_atual).sum(), 2),
}])

alocacao = pd.concat([alocacao, linha_total], ignore_index=True)

print("\n", alocacao)

Sucesso: PETR4.SA
Sucesso: VALE3.SA
Sucesso: ITUB4.SA
Sucesso: BBDC4.SA
Sucesso: BBAS3.SA
Sucesso: ITSA4.SA
Sucesso: WEGE3.SA
Sucesso: RENT3.SA
Sucesso: SUZB3.SA
Sucesso: ABEV3.SA

10 ações carregadas com sucesso.
Geração    1 | melhor fitness: 9.001834
Geração   20 | melhor fitness: 9.138415
Geração   40 | melhor fitness: 9.348420
Geração   60 | melhor fitness: 9.830564
Geração   80 | melhor fitness: 9.858483
Geração  100 | melhor fitness: 10.003644
Geração  120 | melhor fitness: 10.003644
Geração  140 | melhor fitness: 10.118132
Geração  160 | melhor fitness: 10.183762
Geração  180 | melhor fitness: 10.211428
Geração  200 | melhor fitness: 10.211428
Geração  220 | melhor fitness: 10.216030
Geração  240 | melhor fitness: 10.217601
Geração  260 | melhor fitness: 10.217601
Geração  280 | melhor fitness: 10.220477
Geração  300 | melhor fitness: 10.220477

=== MELHOR CARTEIRA ENCONTRADA (via GA) ===
PETR4.SA: 25.36%
VALE3.SA: 3.71%
ITUB4.SA: 20.77%
BBDC4.SA: 3.60%
BBAS3.SA: 4.22%
ITSA4.SA